# Online Retail II: EDA и прогноз дневной выручки

Датасет: транзакции британского интернет-магазина за 2009–2011 годы (UCI ML Repository).
Цель: полный цикл — профиль, очистка, EDA, признаки, `Pipeline` с `ColumnTransformer`, проверка утечек и сравнение моделей (`GradientBoosting`, `HistGradientBoosting`, `LightGBM`).

Таргет обучается в логарифме (`log1p`), метрики считаются на исходной шкале.

In [1]:
from pathlib import Path
import json

import joblib
import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import holidays

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

RAW = ROOT / "data/raw"
PROCESSED = ROOT / "data/processed"
OUTPUTS = ROOT / "outputs"
FIGURES = OUTPUTS / "figures"
MODELS = ROOT / "models"
for folder in (PROCESSED, OUTPUTS, FIGURES, MODELS):
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 25)
print("project root:", ROOT)

project root: D:\ML\Projects\kwork\eda_model


In [2]:
full_path = RAW / "online_retail_ii.csv"
sample_path = RAW / "online_retail_ii_sample.csv"
path = full_path if full_path.exists() else sample_path

df = pd.read_csv(path, parse_dates=["InvoiceDate"])
print("file:", path.name, "shape:", df.shape)
df.head()

file:

 online_retail_ii_sample.csv shape: (100000, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Year
0,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,Year 2009-2010
1,489436,21754,HOME BUILDING BLOCK WORD,3,2009-12-01 09:06:00,5.95,13078.0,United Kingdom,Year 2009-2010
2,489437,22143,CHRISTMAS CRAFT HEART DECORATIONS,6,2009-12-01 09:08:00,2.10,15362.0,United Kingdom,Year 2009-2010
3,489437,22130,PARTY CONE CHRISTMAS DECORATION,12,2009-12-01 09:08:00,0.85,15362.0,United Kingdom,Year 2009-2010
4,489437,21912,VINTAGE SNAKES & LADDERS,4,2009-12-01 09:08:00,3.75,15362.0,United Kingdom,Year 2009-2010


## 1. Профиль и очистка

Считаем пропуски, дубликаты и «грязные» записи, затем чистим по шагам с подсчётом потерь.

In [3]:
duplicates = df.duplicated().sum()
cancelled = df["Invoice"].astype(str).str.startswith("C").sum()
negative_qty = (df["Quantity"] < 0).sum()
zero_price = (df["Price"] <= 0).sum()
print(f"duplicates: {duplicates}, cancelled: {cancelled}, negative qty: {negative_qty}, price<=0: {zero_price}")

duplicates: 105, cancelled: 1879, negative qty: 2216, price<=0: 565


In [4]:
raw_rows = len(df)
work = df.copy()

step1 = work[~work["Invoice"].astype(str).str.startswith("C")]
step2 = step1[step1["Quantity"] > 0]
step3 = step2[step2["Price"] > 0]
step4 = step3.dropna(subset=["Customer ID"])
cleaned = step4.drop_duplicates().copy()
cleaned["Revenue"] = cleaned["Quantity"] * cleaned["Price"]

print(f"raw: {raw_rows:,} -> dedup: {len(cleaned):,} ({len(cleaned) / raw_rows * 100:.1f}%)")
cleaned.to_csv(PROCESSED / "retail_cleaned.csv", index=False)
print("saved:", PROCESSED / "retail_cleaned.csv")

raw: 100,000 -> dedup: 75,412 (75.4%)


saved: D:\ML\Projects\kwork\eda_model\data\processed\retail_cleaned.csv


## 2. EDA

Выручка по месяцам показывает сезонность, страны — перекос рынков.

In [5]:
monthly = cleaned.set_index("InvoiceDate")["Revenue"].resample("ME").sum()

fig, ax = plt.subplots(figsize=(11, 4))
monthly.plot(ax=ax, marker="o", linewidth=1.5)
ax.set_title("Выручка по месяцам")
ax.set_ylabel("£")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "revenue_by_month.png", dpi=150)
plt.show()

C:\Users\boroz\AppData\Local\Temp\ipykernel_24416\2970011556.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
top_countries = cleaned.groupby("Country")["Revenue"].sum().sort_values(ascending=False).head(8)

fig, ax = plt.subplots(figsize=(8, 4))
top_countries.sort_values().plot.barh(ax=ax, color="#4C78A8")
ax.set_title("Выручка по странам, топ-8")
ax.set_xlabel("£")
fig.tight_layout()
fig.savefig(FIGURES / "top_countries.png", dpi=150)
plt.show()
top_countries.to_frame("revenue")

C:\Users\boroz\AppData\Local\Temp\ipykernel_24416\1539208857.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,revenue
Country,
United Kingdom,1333914.921
EIRE,56148.550
Netherlands,52032.570
Germany,38214.100
France,32616.350
Australia,17028.980
Spain,9459.570
Switzerland,9418.140


## 3. Признаки

По дням считаем выручку, заказы, клиентов, средний чек. Все агрегаты транзакций сдвигаем на день назад — иначе получим утечку. Добавляем лаги 1/7/14/28, скользящие средние и разброс, календарь и праздники Великобритании.

In [7]:
daily = cleaned.set_index("InvoiceDate").resample("D").agg(
    Revenue=("Revenue", "sum"),
    Orders=("Invoice", "nunique"),
    Customers=("Customer ID", "nunique"),
    Items=("Quantity", "sum"),
    Products=("StockCode", "nunique"),
)
daily = daily[daily["Revenue"] > 0].copy()
daily["AvgBasket"] = daily["Revenue"] / daily["Orders"]

daily["dow"] = daily.index.dayofweek.astype("category")
daily["month"] = daily.index.month.astype("category")
daily["day"] = daily.index.day
daily["week"] = daily.index.isocalendar().week.astype(int)
daily["is_weekend"] = (daily.index.dayofweek >= 5).astype(int)

uk_holidays = holidays.UK(years=range(daily.index.min().year, daily.index.max().year + 1))
daily["is_holiday"] = daily.index.isin(list(uk_holidays)).astype(int)

for lag in (1, 7, 14, 28):
    daily[f"lag{lag}"] = daily["Revenue"].shift(lag)
for lag in (1, 7):
    daily[f"orders_lag{lag}"] = daily["Orders"].shift(lag)
    daily[f"customers_lag{lag}"] = daily["Customers"].shift(lag)
daily["basket_lag1"] = daily["AvgBasket"].shift(1)

daily["roll7"] = daily["Revenue"].shift(1).rolling(7).mean()
daily["roll14"] = daily["Revenue"].shift(1).rolling(14).mean()
daily["roll28"] = daily["Revenue"].shift(1).rolling(28).mean()
daily["roll_std7"] = daily["Revenue"].shift(1).rolling(7).std()
daily["ratio_lag1_roll7"] = daily["lag1"] / daily["roll7"]

daily = daily.dropna()
print("days:", len(daily), daily.index.min().date(), "-", daily.index.max().date())
daily.head()

days: 576 2010-01-12 - 2011-12-09


,Revenue,Orders,Customers,Items,Products,AvgBasket,dow,month,day,week,is_weekend,is_holiday,...,lag14,lag28,orders_lag1,customers_lag1,orders_lag7,customers_lag7,basket_lag1,roll7,roll14,roll28,roll_std7,ratio_lag1_roll7
InvoiceDate,,,,,,,,,,,,,,,,,,,,,,,,,
2010-01-12,4695.27,22,18,1903,82,213.421364,1,1,12,2,0,0,...,3481.24,3986.26,29.0,26.0,17.0,16.0,81.487241,2541.964286,2189.977857,2959.119643,1918.714777,0.929647
2010-01-13,913.67,14,14,608,35,65.262143,2,1,13,2,0,0,...,2536.11,6314.53,22.0,18.0,25.0,25.0,213.421364,3015.741429,2276.694286,2984.441429,1991.707352,1.556921
2010-01-14,2695.72,42,38,1274,118,64.183810,3,1,14,2,0,0,...,2328.32,6903.99,14.0,14.0,7.0,7.0,65.262143,2952.542857,2160.805714,2791.553571,2059.028593,0.309452
2010-01-15,950.24,26,21,540,47,36.547692,4,1,15,2,0,0,...,1366.48,3149.91,42.0,38.0,29.0,28.0,64.183810,3292.604286,2187.048571,2641.258214,1719.445424,0.818720
2010-01-17,2298.07,29,29,1243,90,79.243793,6,1,17,2,1,0,...,1951.29,1124.66,26.0,21.0,31.0,30.0,36.547692,2613.191429,2157.317143,2562.698571,1536.733910,0.363632


## 4. Проверка утечек

Проверяем хронологию, отсутствие таргета в признаках, корректность лагов и то, что препроцессор обучается только на train.

In [8]:
cat_features = ["dow", "month"]
num_features = [
    "day", "week", "is_weekend", "is_holiday",
    "lag1", "lag7", "lag14", "lag28",
    "orders_lag1", "orders_lag7", "customers_lag1", "customers_lag7", "basket_lag1",
    "roll7", "roll14", "roll28", "roll_std7", "ratio_lag1_roll7",
]
features = cat_features + num_features

X = daily[features]
y = daily["Revenue"]

split = int(len(daily) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

assert "Revenue" not in features
assert X_train.index.max() < X_test.index.min()
assert daily["lag1"].iloc[1] == daily["Revenue"].iloc[0]
assert daily["lag28"].iloc[28] == daily["Revenue"].iloc[0]
assert not X.isna().any().any()

print(f"train: {X_train.index.min().date()} — {X_train.index.max().date()} ({len(X_train)})")
print(f"test:  {X_test.index.min().date()} — {X_test.index.max().date()} ({len(X_test)})")
print("проверки утечки пройдены")

train: 2010-01-12 — 2011-07-26 (460)
test:  2011-07-27 — 2011-12-09 (116)
проверки утечки пройдены


## 5. Модели и CV

Таргет учим в `log1p`, обратно — `expm1`, метрики на исходной шкале. Внутри train — `TimeSeriesSplit` на 5 фолдов, чтобы сравнивать модели не по одному отрезку. Сравниваем `GradientBoosting`, `HistGradientBoosting` и `LightGBM`; препроцессинг одинаковый: `OneHotEncoder` + `StandardScaler` в `ColumnTransformer`.

In [9]:
preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
    ("num", StandardScaler(), num_features),
])

models = {
    "gradient_boosting": GradientBoostingRegressor(random_state=42),
    "hist_gradient_boosting": HistGradientBoostingRegressor(random_state=42, max_iter=400, learning_rate=0.05),
    "lightgbm": lgb.LGBMRegressor(
        n_estimators=400, learning_rate=0.03, num_leaves=15,
        min_child_samples=10, subsample=0.9, colsample_bytree=0.9,
        random_state=42, verbose=-1,
    ),
}

def fit_model(estimator, X_tr, y_tr):
    pipe = Pipeline([("prep", preprocess), ("model", estimator)])
    pipe.fit(X_tr, np.log1p(y_tr))
    return pipe

def predict(pipe, X_te):
    return np.expm1(pipe.predict(X_te))

def metrics(y_true, y_pred):
    return {
        "mae": round(float(mean_absolute_error(y_true, y_pred)), 2),
        "rmse": round(float(mean_squared_error(y_true, y_pred)) ** 0.5, 2),
        "mape": round(float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100), 2),
        "r2": round(float(r2_score(y_true, y_pred)), 3),
    }

In [10]:
tscv = TimeSeriesSplit(n_splits=5)
cv_results = {}
for name, estimator in models.items():
    fold_mae = []
    for train_idx, valid_idx in tscv.split(X_train):
        pipe = fit_model(estimator, X_train.iloc[train_idx], y_train.iloc[train_idx])
        fold_mae.append(mean_absolute_error(y_train.iloc[valid_idx], predict(pipe, X_train.iloc[valid_idx])))
    cv_results[name] = {"mae_mean": round(float(np.mean(fold_mae)), 2), "mae_std": round(float(np.std(fold_mae)), 2)}
    print(name, cv_results[name])

gradient_boosting {'mae_mean': 992.91, 'mae_std': 254.83}


hist_gradient_boosting {'mae_mean': 965.82, 'mae_std': 241.81}


lightgbm {'mae_mean': 958.23, 'mae_std': 249.49}


In [11]:
results = {"baseline_lag1": metrics(y_test, X_test["lag1"])}
fitted = {}
for name, estimator in models.items():
    pipe = fit_model(estimator, X_train, y_train)
    fitted[name] = pipe
    results[name] = metrics(y_test, predict(pipe, X_test))

table = pd.DataFrame(results).T
table["cv_mae_mean"] = table.index.map(lambda n: cv_results.get(n, {}).get("mae_mean"))
table

,mae,rmse,mape,r2,cv_mae_mean
baseline_lag1,1584.08,2056.15,59.40,-0.535,NaN
gradient_boosting,1159.94,1513.26,40.76,0.169,992.91
hist_gradient_boosting,1230.90,1617.50,45.64,0.050,965.82
lightgbm,1163.42,1515.70,42.01,0.166,958.23


In [12]:
scaler = fitted["lightgbm"].named_steps["prep"].named_transformers_["num"]
print("scaler.mean_ == train means:", np.allclose(scaler.mean_, X_train[num_features].mean().to_numpy()))
print("scaler.mean_ == full means: ", np.allclose(scaler.mean_, X[num_features].mean().to_numpy()))

best_name = min(cv_results, key=lambda k: cv_results[k]["mae_mean"])
print("best by CV MAE:", best_name, cv_results[best_name])

scaler.mean_ == train means: True
scaler.mean_ == full means:  False
best by CV MAE: lightgbm {'mae_mean': 958.23, 'mae_std': 249.49}


## 6. Результаты

Лучшую модель сохраняем вместе с препроцессором, предсказания и метрики — в `outputs/`.

In [13]:
best_pipe = fitted[best_name]
summary = {
    "variants": results,
    "cv": cv_results,
    "best": best_name,
    "test_days": int(len(y_test)),
}
(OUTPUTS / "metrics.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

pred = predict(best_pipe, X_test)
predictions = pd.DataFrame({"date": y_test.index, "actual": y_test.values, "predicted": pred})
predictions.to_csv(OUTPUTS / "predictions.csv", index=False)
joblib.dump(best_pipe, MODELS / "pipeline.pkl")
print("saved pipeline:", MODELS / "pipeline.pkl")

saved pipeline: D:\ML\Projects\kwork\eda_model\models\pipeline.pkl


In [14]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y_test.index, y_test.values, label="факт", linewidth=1.5)
ax.plot(y_test.index, pred, label="прогноз", linewidth=1.5, alpha=0.85)
ax.set_title(f"Дневная выручка: факт и прогноз ({best_name})")
ax.set_ylabel("£")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "actual_vs_predicted.png", dpi=150)
plt.show()

C:\Users\boroz\AppData\Local\Temp\ipykernel_24416\186083811.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
prep = best_pipe.named_steps["prep"]
names = prep.get_feature_names_out()
importance = pd.Series(best_pipe.named_steps["model"].feature_importances_, index=names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
importance.head(10).sort_values().plot.barh(ax=ax, color="#72B7B2")
ax.set_title("Важность признаков")
fig.tight_layout()
fig.savefig(FIGURES / "feature_importance.png", dpi=150)
plt.show()
importance.head(10).to_frame("importance")

C:\Users\boroz\AppData\Local\Temp\ipykernel_24416\2555576829.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,importance
num__roll28,456
num__lag14,445
num__lag28,413
num__roll7,389
num__lag7,387
num__week,372
num__roll14,364
num__lag1,331
num__basket_lag1,323
num__day,299


## 7. Выводы

- Утечек нет: тест после train, таргет не в признаках, лаги сдвинуты, препроцессор обучен только на train.
- Логарифм таргета и расширенные лаги (14/28 дней, скользящие окна и разброс, агрегаты заказов и клиентов) дают основной прирост относительно первой версии.
- Сравнение моделей идёт и по holdout, и по `TimeSeriesSplit`: один отрезок может случайно совпасть с сезоном.
- `LightGBM` обычно выигрывает по MAE при том же препроцессинге; деревья не чувствительны к масштабированию, поэтому `StandardScaler` здесь — для единообразия пайплайна, а не ради метрик.
- Ограничения: один магазин, нет промо, рекламы и цен конкурентов, горизонт — один день.

### Что дальше

- добавить промо-календарь и праздничные распродажи;
- прогноз на неделю вперёд с рекурсивными лагами;
- отдельные модели по категориям товаров.
